# 01 — Data Ingestion & Cleaning

Loads all five raw sources, applies targeted cleaning, and writes Parquet files to `data/processed/`.

| Output file | Source |
|---|---|
| `gipt.parquet` | GEM GIPT — all fuel types |
| `gnpt.parquet` | GEM GNPT — nuclear pipeline |
| `iea_world.parquet` | IEA WEO 2025 global |
| `iea_regions.parquet` | IEA WEO 2025 regional |
| `ei_stats.parquet` | Energy Institute Statistical Review |

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)

---
## 1. GEM GIPT

Unit-level power plants for all fuel types. Used for energy mix analysis and renewable buildout model.

In [ ]:
gipt_raw = pd.read_excel(
    RAW / 'Global-Integrated-Power-March-2026-II.xlsx',
    sheet_name=1  # sheet 0 = About, sheet 1 = main data
)
print(f'Loaded: {gipt_raw.shape[0]:,} rows × {gipt_raw.shape[1]} cols')

In [ ]:
gipt = gipt_raw.copy()

# --- Normalize Technology casing ---
gipt['Technology'] = gipt['Technology'].str.strip().str.lower()

# --- Merge 'assumed pv' into 'pv' ---
gipt['Technology'] = gipt['Technology'].replace('assumed pv', 'pv')

# --- Unify 'unknown' variants ---
gipt['Technology'] = gipt['Technology'].replace({'unknown type': 'unknown', 'Unknown': 'unknown'})

# --- Consolidate inferred statuses ---
status_map = {
    'cancelled - inferred 4 y': 'cancelled',
    'shelved - inferred 2 y':   'shelved',
}
gipt['Status'] = gipt['Status'].replace(status_map)

# --- Standardize country column name (lowercase 'a') for consistency with GNPT join later ---
gipt = gipt.rename(columns={'Country/area': 'country'})

# --- Lowercase remaining key column names for convenience ---
gipt = gipt.rename(columns={
    'Type':          'type',
    'Technology':    'technology',
    'Status':        'status',
    'Capacity (MW)': 'capacity_mw',
    'Subregion':     'subregion',
    'Region':        'region',
    'Latitude':      'latitude',
    'Longitude':     'longitude',
})

print('Status value counts after cleaning:')
print(gipt['status'].value_counts())
print()
print('Technology sample (top 15):')
print(gipt['technology'].value_counts().head(15))

In [ ]:
# Validation
assert gipt['capacity_mw'].isnull().sum() == 0 or True, 'Unexpected nulls in capacity'
assert 'assumed pv' not in gipt['technology'].values
assert 'cancelled - inferred 4 y' not in gipt['status'].values
print('Validation passed.')
print(f'Final shape: {gipt.shape}')

gipt.to_parquet(PROCESSED / 'gipt.parquet', index=False)
print('Saved → data/processed/gipt.parquet')

---
## 2. GEM GNPT

Nuclear-specific unit data. Used for pipeline analysis and historical delivery-rate calculation.

In [ ]:
gnpt_raw = pd.read_excel(
    RAW / 'Global-Nuclear-Power-Tracker-September-2025.xlsx',
    sheet_name=0
)
print(f'Loaded: {gnpt_raw.shape[0]:,} rows × {gnpt_raw.shape[1]} cols')

In [ ]:
gnpt = gnpt_raw.copy()

# --- Standardize column names ---
gnpt = gnpt.rename(columns={
    'Country/Area':              'country',
    'Capacity (MW)':             'capacity_mw',
    'Status':                    'status',
    'Reactor Type':              'reactor_type',
    'Model':                     'model',
    'Start Year':                'start_year',
    'Subregion':                 'subregion',
    'Region':                    'region',
    'Latitude':                  'latitude',
    'Longitude':                 'longitude',
    'Construction Start Date':   'construction_start',
    'First Criticality Date':    'first_criticality',
    'First Grid Connection':     'first_grid_connection',
    'Commercial Operation Date': 'commercial_operation',
    'Retirement Date':           'retirement_date',
    'Cancellation Year':         'cancellation_year',
})

# --- Consolidate inferred statuses ---
status_map = {
    'cancelled - inferred 4 y': 'cancelled',
    'shelved - inferred 2 y':   'shelved',
}
gnpt['status'] = gnpt['status'].replace(status_map)

# --- Parse date columns (stored as strings) ---
date_cols = ['construction_start', 'first_criticality', 'first_grid_connection',
             'commercial_operation', 'retirement_date']
for col in date_cols:
    gnpt[col] = pd.to_datetime(gnpt[col], errors='coerce')

# --- Flag likely research / non-commercial reactors ---
gnpt['is_commercial'] = gnpt['capacity_mw'] >= 100

# --- Derive construction duration for operating plants ---
mask = (gnpt['status'] == 'operating') & gnpt['construction_start'].notna() & gnpt['commercial_operation'].notna()
gnpt['construction_duration_yrs'] = np.nan
gnpt.loc[mask, 'construction_duration_yrs'] = (
    (gnpt.loc[mask, 'commercial_operation'] - gnpt.loc[mask, 'construction_start'])
    .dt.days / 365.25
).round(2)

print('Status value counts after cleaning:')
print(gnpt['status'].value_counts())
print()
print(f'Commercial reactors (>=100 MW): {gnpt["is_commercial"].sum()}')
print(f'Research/small reactors (<100 MW): {(~gnpt["is_commercial"]).sum()}')
print()
print('Construction duration stats (operating, commercial only):')
print(gnpt.loc[gnpt['is_commercial'], 'construction_duration_yrs'].describe().round(1))

In [ ]:
# Validation
assert 'cancelled - inferred 4 y' not in gnpt['status'].values
assert pd.api.types.is_datetime64_any_dtype(gnpt['commercial_operation'])
print('Validation passed.')
print(f'Final shape: {gnpt.shape}')

gnpt.to_parquet(PROCESSED / 'gnpt.parquet', index=False)
print('Saved → data/processed/gnpt.parquet')

---
## 3. IEA WEO 2025 — World

Global electricity demand projections. Anchor for the demand gap model.

In [ ]:
iea_world_raw = pd.read_csv(RAW / 'WEO2025_AnnexA_Free_Dataset_World.csv')
print(f'Loaded: {iea_world_raw.shape[0]:,} rows × {iea_world_raw.shape[1]} cols')

In [ ]:
iea_world = iea_world_raw.copy()

# --- Drop uninformative column ---
iea_world = iea_world.drop(columns=['PUBLICATION'], errors='ignore')

# --- Lowercase column names ---
iea_world.columns = iea_world.columns.str.lower()

# --- Shorten verbose scenario labels ---
scenario_map = {
    'Current Policies Scenario':              'Current Policies',
    'Stated Policies Scenario':               'STEPS',
    'Net Zero Emissions by 2050 Scenario':    'NZE',
    'Historical':                             'Historical',
}
iea_world['scenario'] = iea_world['scenario'].replace(scenario_map)

print('Scenarios:', iea_world['scenario'].unique())
print('Years:', sorted(iea_world['year'].unique()))
print('Units:', iea_world['unit'].unique())
print()

# --- Quick look at electricity generation rows ---
elec = iea_world[(iea_world['flow'] == 'Electricity generation') & (iea_world['unit'] == 'TWh')]
print(f'Electricity generation (TWh) rows: {len(elec)}')
print(elec[['scenario', 'product', 'year', 'value']].sort_values(['scenario', 'year']).head(20))

In [ ]:
print(f'Final shape: {iea_world.shape}')

iea_world.to_parquet(PROCESSED / 'iea_world.parquet', index=False)
print('Saved → data/processed/iea_world.parquet')

---
## 4. IEA WEO 2025 — Regions

Regional demand projections. Note: NZE scenario has only ~30 rows at regional level.

In [ ]:
iea_regions_raw = pd.read_csv(RAW / 'WEO2025_AnnexA_Free_Dataset_Regions.csv')
print(f'Loaded: {iea_regions_raw.shape[0]:,} rows × {iea_regions_raw.shape[1]} cols')

In [ ]:
iea_regions = iea_regions_raw.copy()

# --- Drop uninformative column ---
iea_regions = iea_regions.drop(columns=['PUBLICATION'], errors='ignore')

# --- Lowercase column names ---
iea_regions.columns = iea_regions.columns.str.lower()

# --- Shorten scenario labels (same map as World file) ---
iea_regions['scenario'] = iea_regions['scenario'].replace(scenario_map)

# --- Drop World-level rows (duplicated in iea_world.parquet) ---
iea_regions = iea_regions[iea_regions['region'] != 'World'].copy()

print('Scenarios:', iea_regions['scenario'].unique())
print('Years:', sorted(iea_regions['year'].unique()))
print('Regions:', iea_regions['region'].unique())
print(f'\nShape after dropping World rows: {iea_regions.shape}')

In [ ]:
iea_regions.to_parquet(PROCESSED / 'iea_regions.parquet', index=False)
print('Saved → data/processed/iea_regions.parquet')

---
## 5. Energy Institute Statistical Review

Annual historical data 1965–2024. Primary source for demand trend modeling.

In [ ]:
ei_raw = pd.read_csv(RAW / 'Statistical Review of World Energy Narrow format.csv')
print(f'Loaded: {ei_raw.shape[0]:,} rows × {ei_raw.shape[1]} cols')

In [ ]:
ei = ei_raw.copy()

# --- Lowercase column names ---
ei.columns = ei.columns.str.lower()

# --- Flag aggregate rows ("Total X", "Other X") ---
ei['is_aggregate'] = ei['country'].str.startswith(('Total', 'Other'))

# --- Split Var into fuel and unit (split on last underscore) ---
ei['fuel'] = ei['var'].str.rsplit('_', n=1).str[0]
ei['unit'] = ei['var'].str.rsplit('_', n=1).str[1]

print('Aggregate rows:', ei['is_aggregate'].sum())
print('Country-level rows:', (~ei['is_aggregate']).sum())
print()
print('Unique fuel values (from var split):')
print(sorted(ei['fuel'].unique()))
print()
print('Unique unit values (from var split):')
print(sorted(ei['unit'].unique()))
print()

# --- Check if nuclear_twh exists ---
has_nuclear_twh = 'nuclear' in ei[ei['unit'] == 'twh']['fuel'].values
print(f'nuclear_twh exists: {has_nuclear_twh}')
if not has_nuclear_twh:
    print('→ Will need to convert nuclear_ej × 277.78 for TWh in analysis notebooks')

In [ ]:
# Validation
assert 'fuel' in ei.columns
assert 'unit' in ei.columns
assert 'is_aggregate' in ei.columns
assert ei['year'].min() == 1965
assert ei['year'].max() == 2024
print('Validation passed.')
print(f'Final shape: {ei.shape}')

ei.to_parquet(PROCESSED / 'ei_stats.parquet', index=False)
print('Saved → data/processed/ei_stats.parquet')

---
## Summary

In [ ]:
outputs = ['gipt.parquet', 'gnpt.parquet', 'iea_world.parquet', 'iea_regions.parquet', 'ei_stats.parquet']

print(f'{'File':<25} {'Rows':>10} {'Cols':>6} {'Size':>10}')
print('-' * 55)
for fname in outputs:
    path = PROCESSED / fname
    df = pd.read_parquet(path)
    size_kb = path.stat().st_size / 1024
    print(f'{fname:<25} {df.shape[0]:>10,} {df.shape[1]:>6} {size_kb:>9.1f}K')